In [ ]:
import os

# Reactiveaza Keras 2, ca sa functioneze tf.layers.dense
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import numpy as np

In [ ]:
import gymnasium as gym

# Importam TensorFlow in modul de compatibilitate cu versiunea 1.
# "disable_v2_behavior" opreste executia imediata (eager execution).
# Liniile de mai jos NU calculeaza nimic cand sunt executate, ci doar CONSTRUIESC un graf de operatii. Calculul propriu-zis se face mai tarziu, la apelul sess.run(...).
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

Instructions for updating:
non-resource variables are not supported in the long term


In [ ]:
def env_reset(e):
    out = e.reset()
    return out[0] if isinstance(out, tuple) else out


def env_step(e, a):
    out = e.step(a)
    if len(out) == 5:
        s, r, terminated, truncated, info = out
        return s, r, terminated or truncated
    s, r, done, info = out
    return s, r, done

In [ ]:
# =============================================================================
# 1. MEDIUL
# =============================================================================
# CartPole: un carucior se misca pe o sina, iar pe el este montat un bat
# articulat. Scopul agentului este sa mentina batul in pozitie verticala.
#
#   Stare   : 4 numere reale — pozitia caruciorului, viteza lui,
#             unghiul batului, viteza unghiulara a batului
#   Actiuni : 2 actiuni discrete — 0 = impinge stanga, 1 = impinge dreapta
#   Recompensa: +1 pentru FIECARE pas in care batul nu a cazut
#             (un episod mai lung = un scor mai bun)
env = gym.make("CartPole-v1")

# Numarul de valori care descriu o stare (4)
state_shape = env.observation_space.shape[0]

# Numarul de actiuni posibile (2)
num_actions = env.action_space.n

In [ ]:
# 2. CASTIGUL DEPRECIAT SI NORMALIZAT
# =============================================================================
# Factorul de depreciere (discount factor).
# Controleaza cata importanta dam recompenselor viitoare fata de cele imediate.
# gamma = 0.95 inseamna ca o recompensa peste 10 pasi valoreaza 0.95^10 ~= 0.60 din valoarea unei recompense obtinute acum.
gamma = 0.95

def discount_and_normalize_rewards(episode_rewards):
    """
    Primeste lista recompenselor dintr-un episod si returneaza, pentru fiecare
    pas, "cat de bine a mers DE ATUNCI INCOLO", intr-o forma normalizata.

    Aceasta valoare este cea cu care vom pondera fiecare actiune la antrenament.
    """
    discounted_rewards = np.zeros(len(episode_rewards), dtype=np.float64)

    # -------------------------------------------------------------------------
    # REWARD-TO-GO
    # -------------------------------------------------------------------------
    # Pentru fiecare pas t vrem suma recompenselor obtinute DUPA momentul t,
    # depreciate:
    #     R_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    #
    # Calculata direct, formula ar necesita o bucla in bucla (lent).
    # Mergand invers, observam relatia de recurenta:
    #     R_t = r_t + gamma * R_{t+1}
    # adica fiecare valoare se obtine din cea urmatoare, deja calculata.
    # O singura trecere prin lista este suficienta.

    reward_to_go = 0.0
    for i in reversed(range(len(episode_rewards))):
        reward_to_go = reward_to_go * gamma + episode_rewards[i]
        discounted_rewards[i] = reward_to_go

    # -------------------------------------------------------------------------
    # NORMALIZAREA (standardizarea)
    # -------------------------------------------------------------------------
    # In CartPole, TOATE recompensele sunt +1. Deci toate valorile reward-to-go
    # sunt pozitive. Daca le-am folosi asa cum sunt, algoritmul ar creste
    # probabilitatea TUTUROR actiunilor — inclusiv a celor proaste — si nu ar
    # invata nimic util.
    #
    # Scazand media, valorile devin centrate in jurul lui zero:
    #     actiuni MAI BUNE decat media episodului -> valoare POZITIVA -> intarite
    #     actiuni MAI SLABE decat media episodului -> valoare NEGATIVA -> slabite
    #
    # Impartirea la abaterea standard aduce valorile la o scara comparabila,
    # ceea ce stabilizeaza antrenamentul.
    discounted_rewards -= np.mean(discounted_rewards)
    std = np.std(discounted_rewards)
    if std > 1e-8:
        discounted_rewards /= std

    return discounted_rewards

In [ ]:
# =============================================================================
# 3. RETEAUA
# =============================================================================
# Un placeholder este un "loc gol" in graf, in care vom turna datele reale mai
# tarziu, prin feed_dict.
#
# [!] Ce inseamna "None" in dimensiuni?
# "None" in dimensiuni = orice numar de randuri. Nu se stie cati pasi va avea episodul — poate 20, poate 500 — asa ca lasam dimensiunea nedeterminata.

# Starile din episod: (numar_de_pasi x 4)
state_ph = tf.placeholder(tf.float32, [None, state_shape], name="state_ph")

# Actiunile executate, codificate one-hot: (numar_de_pasi x 2)
action_ph = tf.placeholder(tf.float32, [None, num_actions], name="action_ph")

# Castigurile depreciate si normalizate: (numar_de_pasi,)
discounted_rewards_ph = tf.placeholder(tf.float32, [None, ], name="discounted_rewards")

# --- Straturile retelei ---

# Strat ascuns cu 32 de neuroni si activare ReLU.
# "dense" = strat complet conectat: fiecare neuron primeste toate intrarile.
layer1 = tf.layers.dense(state_ph, units=32, activation=tf.nn.relu)

# Stratul de iesire. Numarul de neuroni este EGAL cu numarul de actiuni:
# reteaua produce cate un scor brut pentru fiecare actiune posibila.
layer2 = tf.layers.dense(layer1, units=num_actions)

# Softmax transforma scorurile brute in PROBABILITATI:
# toate devin pozitive si insumeaza exact 1.
# Exemplu: [2.0, 1.0] -> [0.73, 0.27]
# Reteaua nu spune "fa actiunea 1", ci "actiunea 0 in 73% din cazuri, actiunea 1 in 27% din cazuri".
prob_dist = tf.nn.softmax(layer2)

In [ ]:
# =============================================================================
# 4. FUNCTIA DE COST
# =============================================================================
# Formula gradientului pe care o implementam:
#
#   grad J(theta) = (1/N) * SUMA [ grad log pi(a_t|s_t) * R_t ]
#
# iar actualizarea parametrilor se face prin ASCENSIUNE de gradient
# (pentru ca MAXIMIZAM castigul, nu minimizam o eroare):
#
#   theta = theta + alpha * grad J(theta)
#
# Problema practica: bibliotecile de deep learning stiu doar sa MINIMIZEZE.
# Inmultim obiectivul cu -1. Maximizarea lui X este acelasi lucru cu minimizarea lui -X.

# Termenul de care avem nevoie este -log pi(a|s), adica minus logaritmul
# probabilitatii pe care reteaua a dat-o actiunii executate efectiv.
# Exact asta calculeaza entropia incrucisata intre iesirea
# retelei si actiunea codificata one-hot.
neg_log_policy = tf.nn.softmax_cross_entropy_with_logits_v2(
    logits=layer2,
    labels=action_ph
)

# Comparatie cu invatarea supervizata obisnuita:
#   - clasificare normala : cost = medie( -log p(eticheta_corecta) )
#   - policy gradient     : cost = medie( -log p(actiunea_executata) * CASTIG )
#
# Singura diferenta este ponderea. Fiecare actiune este tratata ca o "eticheta
# de imitat", dar cu cat de multa convingere depinde de rezultatul obtinut:
#
#   castig POZITIV -> "a mers bine, fa asta mai des"  -> creste probabilitatea
#   castig NEGATIV -> "a mers prost, fa asta mai rar" -> scade probabilitatea
#
# Nu exista aici nicio "eticheta corecta" stiuta dinainte. Mediul ne spune,
# prin recompense, ce merita imitat.
loss = tf.reduce_mean(neg_log_policy * discounted_rewards_ph)

# Optimizatorul Adam, cu rata de invatare 0.01.
# "minimize" calculeaza automat gradientii si actualizeaza parametrii retelei.
train = tf.train.AdamOptimizer(0.01).minimize(loss)

In [14]:
num_iterations = 1000

# Lista pentru urmarirea progresului
returns_log = []

# Sesiunea TensorFlow
sess = tf.Session()

  # Initializeaza parametrii retelei cu valori aleatoare.
sess.run(tf.global_variables_initializer())

for i in range(num_iterations):

    # Liste in care adunam tot ce se intampla in episodul curent.
    episode_states, episode_actions, episode_rewards = [], [], []

    done = False
    state = env_reset(env)
    Return = 0     # castigul total al episodului (doar pentru afisare)

    # ---------------------------------------------------------------------
    # FAZA 1 — jucam un episod COMPLET, fara sa invatam nimic inca
    # ---------------------------------------------------------------------
    # Aceasta este diferenta structurala fata de Q-learning si DQN, care se actualizeaza la fiecare pas. REINFORCE nu POATE face asta: are nevoie de castigul R_t, care nu se cunoaste pana la finalul episodului.
    while not done:

        # Reteaua asteapta un lot de exemple, nu un exemplu singur.
        # Transformam vectorul de 4 valori intr-o matrice 1x4.
        state = np.reshape(state, [1, state_shape])

        # Trecem starea prin retea si obtinem distributia de probabilitate.
        pi = sess.run(prob_dist, feed_dict={state_ph: state})

        # ESANTIONAM din distributie: daca reteaua spune
        # [0.7, 0.3], vom alege actiunea 0 in 70% din cazuri si actiunea 1 in 30% din cazuri.
        #
        # De aceea in acest algoritm NU exista epsilon-greedy.
        # Politica fiind stocastica, exploreaza prin natura ei. Pe masura ce invata, distributia devine tot mai ascutita si explorarea scade de la sine.
        #
        # pi.ravel() transforma matricea 1x2 intr-un vector de 2 elemente,
        # forma ceruta de parametrul p= al functiei.
        a = np.random.choice(range(pi.shape[1]), p=pi.ravel())

        # Executam actiunea in mediu.
        next_state, reward, done = env_step(env, a)

        Return += reward

        # Codificare "one-hot": transformam indicele actiunii intr-un
        #     vector cu 1 pe pozitia respectiva si 0 in rest.
        #        actiunea 0 -> [1, 0]
        #        actiunea 1 -> [0, 1]
        #     Aceasta este forma ceruta de functia de entropie incrucisata.
        action = np.zeros(num_actions)
        action[a] = 1

        # Memoram pasul.
        episode_states.append(state)
        episode_actions.append(action)
        episode_rewards.append(reward)

        state = next_state

    # Calculam, pentru fiecare pas, cat de bine a mers de atunci incolo.
    discounted_rewards = discount_and_normalize_rewards(episode_rewards)

    # np.vstack lipeste toate matricele 1x4 intr-o singura matrice de forma (numar_de_pasi x 4). Idem pentru actiuni. Antrenam pe TOT episodul deodata, intr-un singur pas de gradient.
    feed_dict = {
        state_ph:              np.vstack(episode_states),
        action_ph:             np.vstack(episode_actions),
        discounted_rewards_ph: discounted_rewards
    }

    # O SINGURA actualizare a parametrilor, pe baza intregului episod.
    # Dupa aceasta actualizare, datele episodului devin INUTILIZABILE si se arunca. Algoritmul este "on-policy": datele trebuie sa provina de la politica CURENTA.
    loss_, _ = sess.run([loss, train], feed_dict=feed_dict)

    returns_log.append(Return)

    # Afisam progresul din 10 in 10 iteratii.
    # "Return" este numarul de pasi in care batul a stat in picioare. Daca algoritmul invata, aceasta valoare trebuie sa CREASCA.
    if i % 10 == 0:
        media = np.mean(returns_log[-20:])
        print("Iteration:{}, Return: {}, Medie ultimele 20: {:.1f}".format(
            i, Return, media))


Iteration:0, Return: 16.0, Medie ultimele 20: 16.0
Iteration:10, Return: 26.0, Medie ultimele 20: 26.4
Iteration:20, Return: 48.0, Medie ultimele 20: 39.9
Iteration:30, Return: 103.0, Medie ultimele 20: 56.3
Iteration:40, Return: 193.0, Medie ultimele 20: 72.5
Iteration:50, Return: 56.0, Medie ultimele 20: 79.2
Iteration:60, Return: 90.0, Medie ultimele 20: 75.8
Iteration:70, Return: 444.0, Medie ultimele 20: 148.8
Iteration:80, Return: 500.0, Medie ultimele 20: 271.0
Iteration:90, Return: 415.0, Medie ultimele 20: 322.8
Iteration:100, Return: 500.0, Medie ultimele 20: 375.9
Iteration:110, Return: 373.0, Medie ultimele 20: 450.4
Iteration:120, Return: 500.0, Medie ultimele 20: 439.4
Iteration:130, Return: 500.0, Medie ultimele 20: 450.6
Iteration:140, Return: 500.0, Medie ultimele 20: 497.4
Iteration:150, Return: 149.0, Medie ultimele 20: 473.1
Iteration:160, Return: 427.0, Medie ultimele 20: 457.1
Iteration:170, Return: 500.0, Medie ultimele 20: 455.9
Iteration:180, Return: 500.0, Med

## Testare

In [15]:
def choose_action(state, greedy=True):
  # Trece starea prin reteaua antrenata si returneaza o actiune.
  p = sess.run(prob_dist, feed_dict={state_ph: np.reshape(state, [1, state_shape])})
    # DOUA MODURI DE A ALEGE:
    #
    #   greedy=True  -> argmax: alegem mereu actiunea cea mai probabila. Se testeaza ce a invatat politica.
    #
    #   greedy=False -> esantionare din distributie, exact ca la antrenament. Cum se COMPORTA agentul in timpul invatarii.
    #
    # La testare = greedy. Daca scorul greedy e mult mai bun decat cel esantionat, inseamna ca politica stie raspunsul corect, dar inca exploreaza mult — adica mai are de invatat.
  return int(np.argmax(p)) if greedy else int(np.random.choice(num_actions, p=p.ravel()))

In [16]:
def evaluate(n_episodes = 20, greedy=True, random_policy = True):
  e = gym.make('CartPole-v1')
  scores = []
  for _ in range(n_episodes):
    out = e.reset()
    s = out[0] if isinstance(out, tuple) else out
    done, total = False, 0
    while not done:
      out = e.step(a)
      if len(out) == 5:
        s, r, term, trunc, _ = out
        done = term or trunc
      else:
        s, r, done, _ = out
      total += r

    scores.append(total)

  e.close()
  return np.array(scores)


sc_greedy = evaluate(20, greedy = True)
sc_sample = evaluate(20, greedy=False)
sc_random = evaluate(20, random_policy=True)


In [17]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
# render_mode="rgb_array" returneaza fiecare cadru ca imagine (matrice depixeli), in loc sa deschida o fereastra pe ecran.
e = gym.make("CartPole-v1", render_mode='rgb_array')
out = e.reset()
s = out[0] if isinstance(out, tuple) else out

frames, done = [], False
while not done and len(frames) < 500:
  frames.append(e.render())
  out=e.step(choose_action(s, greedy=True))
  if len(out) == 5:
        s, r, term, trunc, _ = out
        done = term or trunc
  else:
    s, r, done, _ = out

e.close()

fig = plt.figure(figsize=(8, 5))
plt.axis("off")
img = plt.imshow(frames[0])

def update(i):
  img.set_array(frames[i])
  return [img]

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=30, blit=True)

plt.close()
HTML(anim.to_jshtml())